In [1]:
import os

# The path you identified
DATASET_PATH = '/content/drive/.shortcut-targets-by-id/1pCrT-i-OuLmydhrMGwT2JJ-n0AOaAKDQ/ColabNotebooks/document-classifier/data/scanned_docs'

if os.path.exists(DATASET_PATH):
    subfolders = [f for f in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, f))]
    print(f"✅ Success! Path is accessible.")
    print(f"Found {len(subfolders)} classes: {subfolders}")
else:
    print("❌ Path not found. Double-check if the folder is still 'Added to Shortcut' in your Drive.")

✅ Success! Path is accessible.
Found 10 classes: ['ADVE', 'Email', 'Form', 'Letter', 'News', 'Note', 'Resume', 'Report', 'Scientific', 'Memo']


In [2]:
import os

TEXT_PATH = '/content/drive/.shortcut-targets-by-id/1AhfnCMVwculYjFHcvMlYXWIEvtKo0i3y/document_classifier/cache/text'

if os.path.exists(TEXT_PATH):
    files = os.listdir(TEXT_PATH)
    txt_files = [f for f in files if f.endswith('.txt')]
    print(f"✅ OCR Path accessible. Found {len(txt_files)} text files.")
    print("Samples of OCR filenames:")
    print(txt_files[:10]) # Show first 10 files
else:
    print("❌ OCR Path NOT found. Check the ID in the path string.")

✅ OCR Path accessible. Found 3482 text files.
Samples of OCR filenames:
['1003044160-e.txt', '1003044181.txt', '1003044360-a.txt', '1003044520-b.txt', '1003044581-a.txt', '10030463.txt', '10030834.txt', '1003099991.txt', '10031067.txt', '10031147.txt']


In [3]:
import os
import torch

# --- DIRECTORIES ---
CSV_PATH = '/content/drive/.shortcut-targets-by-id/1AhfnCMVwculYjFHcvMlYXWIEvtKo0i3y/document_classifier/cache/all_text_dataset.csv'
# Target path for models
SAVE_DIR = '/content/drive/.shortcut-targets-by-id/1pCrT-i-OuLmydhrMGwT2JJ-n0AOaAKDQ/ColabNotebooks/document-classifier/models/trained_models'
CHECKPOINT_NAME = 'bert_checkpoint.pth'
os.makedirs(SAVE_DIR, exist_ok=True)

# --- HYPERPARAMETERS ---
MODEL_NAME = 'bert-base-uncased'
MAX_LEN = 512
BATCH_SIZE = 16 
EPOCHS = 10
LR = 2e-5

# Device Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Configuration Complete. Using device: {device}")

Configuration Complete. Using device: cuda


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

# Load and Force String Type
df = pd.read_csv(CSV_PATH)
df['text'] = df['text'].fillna('[NO_TEXT_DETECTED]').astype(str)

# Label Encoding
le = LabelEncoder()
df['label_idx'] = le.fit_transform(df['label'])
num_classes = len(le.classes_)

# Stratified Split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_idx'])

class DocumentTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts.to_list()
        self.labels = labels.to_list()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, item):
        encoding = self.tokenizer(self.texts[item], add_special_tokens=True, max_length=self.max_len,
                                  padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[item], dtype=torch.long)
        }

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_loader = DataLoader(DocumentTextDataset(train_df['text'], train_df['label_idx'], tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(DocumentTextDataset(val_df['text'], val_df['label_idx'], tokenizer, MAX_LEN), batch_size=BATCH_SIZE)

print(f"Data Prepared. Classes: {list(le.classes_)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Data Prepared. Classes: ['ADVE', 'Email', 'Form', 'Letter', 'Memo', 'News', 'Note', 'Report', 'Resume', 'Scientific']


In [5]:
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Initialize Model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_classes).to(device)

# Initialize Optimizer
optimizer = AdamW(model.parameters(), lr=LR)

# Initialize Plateau Scheduler (Removed 'verbose' argument for compatibility)
plateau_scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=1)

start_epoch = 0
best_val_acc = 0
checkpoint_path = os.path.join(SAVE_DIR, CHECKPOINT_NAME)

# RESUME LOGIC
if os.path.exists(checkpoint_path):
    print("Found existing checkpoint. Loading status...")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_acc = checkpoint['best_val_acc']
    print(f"Resuming from Epoch {start_epoch}")
else:
    print("No checkpoint found. Starting fresh training.")

# Linear Warmup Scheduler
# We calculate steps based on the remaining epochs
total_steps = len(train_loader) * (EPOCHS - start_epoch)
warmup_scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=0, 
    num_training_steps=total_steps if total_steps > 0 else 1
)

print("Model and Optimizers initialized successfully.")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


No checkpoint found. Starting fresh training.
Model and Optimizers initialized successfully.


In [6]:
from sklearn.metrics import accuracy_score
from tqdm import tqdm

# Settings for Early Stopping
patience_limit = 3
epochs_without_improvement = 0

for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_loss = 0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch in train_loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        warmup_scheduler.step()
        
        train_loss += loss.item()
        train_loop.set_description(f"Loss: {loss.item():.4f}")

    # Validation Phase
    model.eval()
    val_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids, mask, labels = batch['input_ids'].to(device), batch['attention_mask'].to(device), batch['labels'].to(device)
            outputs = model(input_ids, attention_mask=mask, labels=labels)
            val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy()); all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}: Val Acc: {val_acc:.4f} | Val Loss: {avg_val_loss:.4f}")

    # 1. Update the Learning Rate Scheduler (Plateau)
    plateau_scheduler.step(avg_val_loss)

    # 2. Check for Improvement
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        epochs_without_improvement = 0 # Reset counter
        state = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_val_acc': best_val_acc,
            'label_encoder': le,
            'classes': le.classes_
        }
        torch.save(state, checkpoint_path)
        print(f"⭐️ New Best Model Saved (Acc: {val_acc:.4f})")
    else:
        epochs_without_improvement += 1
        print(f"No improvement for {epochs_without_improvement} epoch(s).")

    # 3. Early Stopping Trigger
    if epochs_without_improvement >= patience_limit:
        print(f"🛑 Early stopping triggered! No improvement for {patience_limit} epochs.")
        break

print(f"\nFinal Training Complete. Best Accuracy: {best_val_acc:.4f}")

Loss: 1.5682: 100%|██████████| 175/175 [04:30<00:00,  1.55s/it]


Epoch 1: Val Acc: 0.5696 | Val Loss: 1.3098
⭐️ New Best Model Saved (Acc: 0.5696)


Loss: 0.1420: 100%|██████████| 175/175 [04:48<00:00,  1.65s/it]


Epoch 2: Val Acc: 0.7202 | Val Loss: 0.8357
⭐️ New Best Model Saved (Acc: 0.7202)


Loss: 0.1253: 100%|██████████| 175/175 [04:49<00:00,  1.65s/it]


...

Epoch 8: Val Acc: 0.8336 | Val Loss: 0.6244
⭐️ New Best Model Saved (Acc: 0.8336)


Loss: 0.0072: 100%|██████████| 175/175 [04:48<00:00,  1.65s/it]


Epoch 9: Val Acc: 0.8192 | Val Loss: 0.6634
No improvement for 1 epoch(s).


Loss: 0.0128: 100%|██████████| 175/175 [04:47<00:00,  1.64s/it]


Epoch 10: Val Acc: 0.8350 | Val Loss: 0.6557
⭐️ New Best Model Saved (Acc: 0.8350)

Final Training Complete. Best Accuracy: 0.8350
